# 04 — Evaluation (Session A, hi→mr)

Loads a PEFT adapter and evaluates it on the configured benchmarks
(`ai4bharat/IN22-Gen` gen + `facebook/flores` devtest, `hin_Deva-mar_Deva`)
with sacreBLEU BLEU and chrF++. Set `CONFIG`, `ADAPTER`, `FAMILY` below, then run all cells.

In [ ]:
# Run once per environment (Kaggle-safe).
%pip install -q sacrebleu datasets transformers peft IndicTransToolkit matplotlib pyyaml

In [ ]:
import sys
sys.path.insert(0, "../src")  # repo layout: src/mr_mt; notebook lives in notebooks/

CONFIG = "../configs/base.yaml"   # cell config, e.g. configs/cell_*.yaml
ADAPTER = ""                      # PEFT adapter path or HF id ("" = base model only)
FAMILY = "bodhan"                 # "bodhan" | "indictrans2"

from mr_mt.config import load_config
from mr_mt.evaluate import load_model_for_family

cfg = load_config(CONFIG)
print("benchmarks:", [b.get("name") for b in cfg.get("eval", {}).get("benchmarks", [])])
model, tokenizer = load_model_for_family(cfg, ADAPTER, FAMILY)
print("model loaded for family:", FAMILY)

In [ ]:
from mr_mt.evaluate import main as run_evaluate

# Runs batched generation over cfg.eval.benchmarks and writes
# reports/predictions/<name>_{preds,refs}.txt, reports/metrics.json,
# reports/predictions/samples.md + an experiments.csv (cell=eval) row.
metrics = run_evaluate(["--config", CONFIG, "--adapter", ADAPTER, "--family", FAMILY])
metrics

In [ ]:
import json
from pathlib import Path

print(Path("reports/metrics.json").read_text(encoding="utf-8"))
print("--- samples (first 20) ---")
print(Path("reports/predictions/samples.md").read_text(encoding="utf-8"))

from mr_mt.plots import plot_metric_bars
plot_metric_bars(metrics, "reports/figures/metric_bars.png")
from IPython.display import Image
Image("reports/figures/metric_bars.png")